# NJ + NY commute processing (2021, JT00)

Run the two code cells in order. Only the six real LODES files are used. Outputs are saved beside this notebook in `njny_commute_outputs`. The original NJ-only outputs are preserved.

The study region includes New Jersey and New York State. AUX files are filtered to the two cross-state directions. S000 counts jobs, not observed daily trips or unique people. Shares exclude connections to other states. Geographic units are state × ZCTA, not postal ZIP. All code cells have been checked against the supplied files.

## 1. Map blocks and aggregate four commute directions

Records without a valid ZCTA at either endpoint are saved separately and counted in the audit; they are excluded from ZCTA features rather than assigned to invented regions.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"C:\Users\41240\OneDrive\桌面\MBS\2026 fall\Advanced Analytics\data")
OUTPUT = Path.cwd() / "njny_commute_outputs"
OUTPUT.mkdir(exist_ok=True)

# State codes are derived from Census block IDs, not inferred from ZIP prefixes.
lookups = []
for state in ["nj", "ny"]:
    path = DATA_DIR / f"{state}_xwalk.csv.gz"
    assert path.exists(), f"Missing file: {path}"
    table = pd.read_csv(path, usecols=["tabblk2020", "zcta"], dtype="string")
    lookups.append(table)
xwalk = pd.concat(lookups, ignore_index=True)
assert not xwalk.tabblk2020.duplicated().any(), "Duplicate block IDs in crosswalk."
lookup = xwalk.set_index("tabblk2020")["zcta"]

# (file state, file type, residence state FIPS, workplace state FIPS)
specs = [("nj", "main", "34", "34"), ("ny", "main", "36", "36"),
         ("ny", "aux", "34", "36"), ("nj", "aux", "36", "34")]
keys = ["origin_state", "origin_zcta", "destination_state", "destination_zcta"]
state_names = {"34": "NJ", "36": "NY"}
parts, audits, unmatched_parts = [], [], []

for file_state, file_type, home_state, work_state in specs:
    path = DATA_DIR / f"{file_state}_od_{file_type}_JT00_2021.csv.gz"
    assert path.exists(), f"Missing file: {path}"
    raw_rows = selected_rows = raw_jobs = selected_jobs = candidate_rows = unmapped_jobs = unmapped_rows = 0
    for chunk in pd.read_csv(path, usecols=["h_geocode", "w_geocode", "S000"],
                             dtype={"h_geocode": "string", "w_geocode": "string"},
                             chunksize=250_000):
        assert chunk.h_geocode.str.fullmatch(r"[0-9]{15}").fillna(False).all()
        assert chunk.w_geocode.str.fullmatch(r"[0-9]{15}").fillna(False).all()
        assert chunk.S000.notna().all() and chunk.S000.ge(0).all()
        raw_rows += len(chunk)
        raw_jobs += int(chunk.S000.sum())
        keep = (chunk.h_geocode.str[:2].eq(home_state)
                & chunk.w_geocode.str[:2].eq(work_state))
        subset = chunk.loc[keep].copy()
        if file_type == "main":
            assert keep.all(), "Unexpected state in main file."
        if subset.empty:
            continue
        candidate_rows += len(subset)
        subset["origin_state"] = state_names[home_state]
        subset["destination_state"] = state_names[work_state]
        subset["origin_zcta"] = subset.h_geocode.map(lookup)
        subset["destination_zcta"] = subset.w_geocode.map(lookup)
        mapped = pd.Series(True, index=subset.index)
        for column in ["origin_zcta", "destination_zcta"]:
            valid = (subset[column].str.fullmatch(r"[0-9]{5}").fillna(False)
                     & ~subset[column].isin(["00000", "99999"]))
            mapped &= valid
        if not mapped.all():
            rejected = subset.loc[~mapped].copy()
            rejected["source_file"] = path.name
            unmatched_parts.append(rejected)
            unmapped_rows += len(rejected)
            unmapped_jobs += int(rejected.S000.sum())
        subset = subset.loc[mapped].copy()
        selected_rows += len(subset)
        selected_jobs += int(subset.S000.sum())
        parts.append(subset.groupby(keys, as_index=False).S000.sum())
    audits.append({"source_file": path.name, "direction": f"{state_names[home_state]}->{state_names[work_state]}",
                   "raw_rows": raw_rows, "selected_rows": selected_rows,
                   "excluded_other_state_rows": raw_rows - candidate_rows,
                   "unmapped_zcta_rows": unmapped_rows, "unmapped_zcta_jobs": unmapped_jobs,
                   "raw_jobs": raw_jobs, "selected_jobs": selected_jobs})
    print(f"Finished {path.name}: selected {selected_rows:,} rows, {selected_jobs:,} jobs")

flows = (pd.concat(parts, ignore_index=True).groupby(keys, as_index=False).S000.sum()
         .rename(columns={"S000": "commute_jobs"}))
flows["year"] = 2021
flows["job_type"] = "JT00"
audit = pd.DataFrame(audits)
assert flows.commute_jobs.sum() == audit.selected_jobs.sum()
assert not flows.duplicated(keys).any()
flows.to_csv(OUTPUT / "njny_commute_zcta_flows_2021.csv", index=False, encoding="utf-8-sig")
audit.to_csv(OUTPUT / "njny_commute_processing_audit_2021.csv", index=False, encoding="utf-8-sig")
if unmatched_parts:
    pd.concat(unmatched_parts, ignore_index=True).to_csv(
        OUTPUT / "njny_commute_unmapped_zcta_records_2021.csv", index=False, encoding="utf-8-sig")
display(audit)
display(flows.head(10))
print("Total jobs:", int(flows.commute_jobs.sum()))


Finished nj_od_main_JT00_2021.csv.gz: selected 3,222,070 rows, 3,439,824 jobs
Finished ny_od_main_JT00_2021.csv.gz: selected 7,142,438 rows, 7,897,753 jobs
Finished ny_od_aux_JT00_2021.csv.gz: selected 475,765 rows, 494,304 jobs
Finished nj_od_aux_JT00_2021.csv.gz: selected 184,702 rows, 187,042 jobs


,source_file,direction,raw_rows,selected_rows,excluded_other_state_rows,unmapped_zcta_rows,unmapped_zcta_jobs,raw_jobs,selected_jobs
0,nj_od_main_JT00_2021.csv.gz,NJ->NJ,3222070,3222070,0,0,0,3439824,3439824
1,ny_od_main_JT00_2021.csv.gz,NY->NY,7144618,7142438,0,2180,2225,7899978,7897753
2,ny_od_aux_JT00_2021.csv.gz,NJ->NY,755891,475765,280079,47,47,776631,494304
3,nj_od_aux_JT00_2021.csv.gz,NY->NJ,392310,184702,207536,72,73,397262,187042


,origin_state,origin_zcta,destination_state,destination_zcta,commute_jobs,year,job_type
0,NJ,07001,NJ,07001,250,2021,JT00
1,NJ,07001,NJ,07002,27,2021,JT00
2,NJ,07001,NJ,07003,17,2021,JT00
3,NJ,07001,NJ,07004,10,2021,JT00
4,NJ,07001,NJ,07005,2,2021,JT00
5,NJ,07001,NJ,07006,3,2021,JT00
6,NJ,07001,NJ,07008,123,2021,JT00
7,NJ,07001,NJ,07009,3,2021,JT00
8,NJ,07001,NJ,07011,4,2021,JT00
9,NJ,07001,NJ,07012,9,2021,JT00


Total jobs: 12018923


## 2. Build regional features

Cross-state totals are subsets of external totals. Do not add them together. These are annual structural attributes, not daily disease-pressure features. Re-read output identifiers as strings. Zero denominators produce missing shares.

In [2]:
# External means outside this ZCTA, within the NJ+NY study region.
# Retain state in the key, including for ZCTAs that may cross state boundaries.
home = ["origin_state", "origin_zcta"]
work = ["destination_state", "destination_zcta"]
def totals(table, columns, name):
    result = table.groupby(columns)["commute_jobs"].sum().rename(name)
    result.index.names = ["state", "zcta"]
    return result
same = (flows.origin_state.eq(flows.destination_state)
        & flows.origin_zcta.eq(flows.destination_zcta))
cross_state = flows.origin_state.ne(flows.destination_state)
metrics = [
    totals(flows, home, "resident_jobs_total"),
    totals(flows, work, "workplace_jobs_total"),
    totals(flows.loc[~same], work, "inbound_jobs_external"),
    totals(flows.loc[~same], home, "outbound_jobs_external"),
    totals(flows.loc[same], home, "internal_jobs"),
    totals(flows.loc[cross_state], work, "inbound_jobs_cross_state"),
    totals(flows.loc[cross_state], home, "outbound_jobs_cross_state"),
]
features = pd.concat(metrics, axis=1).fillna(0).astype("int64")
features["external_inbound_share"] = (features.inbound_jobs_external /
    features.workplace_jobs_total.where(features.workplace_jobs_total.gt(0)))
features["external_outbound_share"] = (features.outbound_jobs_external /
    features.resident_jobs_total.where(features.resident_jobs_total.gt(0)))
features["cross_state_inbound_share"] = (features.inbound_jobs_cross_state /
    features.workplace_jobs_total.where(features.workplace_jobs_total.gt(0)))
features["cross_state_outbound_share"] = (features.outbound_jobs_cross_state /
    features.resident_jobs_total.where(features.resident_jobs_total.gt(0)))
assert (features.internal_jobs + features.inbound_jobs_external).eq(features.workplace_jobs_total).all()
assert (features.internal_jobs + features.outbound_jobs_external).eq(features.resident_jobs_total).all()
features = features.reset_index()
features["year"] = 2021
features["job_type"] = "JT00"
features.to_csv(OUTPUT / "njny_commute_zcta_features_2021.csv", index=False, encoding="utf-8-sig")
print("Saved files to:", OUTPUT.resolve())
print("Region counts:")
print(features.groupby("state").size())
display(features.head(10))


Saved files to: C:\Users\41240\OneDrive\文档\GitHub\practicum-project\njny_commute_outputs
Region counts:
state
NJ     598
NY    1826
dtype: int64


,state,zcta,resident_jobs_total,workplace_jobs_total,inbound_jobs_external,outbound_jobs_external,internal_jobs,inbound_jobs_cross_state,outbound_jobs_cross_state,external_inbound_share,external_outbound_share,cross_state_inbound_share,cross_state_outbound_share,year,job_type
0,NJ,07001,7174,10997,10747,6924,250,660,824,0.977267,0.965152,0.060016,0.114859,2021,JT00
1,NJ,07002,30537,14666,10027,25898,4639,2036,7568,0.683690,0.848086,0.138824,0.247831,2021,JT00
2,NJ,07003,25617,12899,11349,24067,1550,496,4788,0.879836,0.939493,0.038453,0.186907,2021,JT00
3,NJ,07004,4100,20386,19989,3703,397,834,626,0.980526,0.903171,0.040910,0.152683,2021,JT00
4,NJ,07005,8197,4679,4157,7675,522,189,713,0.888438,0.936318,0.040393,0.086983,2021,JT00
5,NJ,07006,13349,10548,9418,12219,1130,356,2278,0.892871,0.915349,0.033750,0.170649,2021,JT00
6,NJ,07008,10583,10316,9364,9631,952,846,1129,0.907716,0.910044,0.082009,0.106681,2021,JT00
7,NJ,07009,6512,4198,3849,6163,349,138,1113,0.916865,0.946407,0.032873,0.170915,2021,JT00
8,NJ,07010,10059,2665,2072,9466,593,486,3799,0.777486,0.941048,0.182364,0.377672,2021,JT00
9,NJ,07011,16285,8216,7516,15585,700,348,1707,0.914800,0.957016,0.042356,0.104820,2021,JT00


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT = Path.cwd() / "njny_commute_outputs"

# Read the flow table and preserve leading zeros in ZCTA identifiers.
flows = pd.read_csv(
    OUTPUT / "njny_commute_zcta_flows_2021.csv",
    dtype={
        "origin_state": "string",
        "origin_zcta": "string",
        "destination_state": "string",
        "destination_zcta": "string"
    }
)

# Exclude connections within the same state–ZCTA.
# The original flow table remains unchanged.
is_internal = (
    flows["origin_state"].eq(flows["destination_state"])
    & flows["origin_zcta"].eq(flows["destination_zcta"])
)

weights = flows.loc[~is_internal].copy()

# Calculate total external inbound jobs for each destination.
inbound_total = weights.groupby(
    ["destination_state", "destination_zcta"]
)["commute_jobs"].transform("sum")

# Calculate total external outbound jobs for each origin.
outbound_total = weights.groupby(
    ["origin_state", "origin_zcta"]
)["commute_jobs"].transform("sum")

# Confirm that all denominators are positive.
assert inbound_total.gt(0).all()
assert outbound_total.gt(0).all()

# Inbound weight: this connection's share of the destination's external inflow.
weights["inbound_weight"] = weights["commute_jobs"] / inbound_total

# Outbound weight: this connection's share of the origin's external outflow.
weights["outbound_weight"] = weights["commute_jobs"] / outbound_total

# Check that inbound weights sum to 1 for each destination.
inbound_check = weights.groupby(
    ["destination_state", "destination_zcta"]
)["inbound_weight"].sum()

# Check that outbound weights sum to 1 for each origin.
outbound_check = weights.groupby(
    ["origin_state", "origin_zcta"]
)["outbound_weight"].sum()

assert np.allclose(inbound_check, 1)
assert np.allclose(outbound_check, 1)

# Save the weights as a new file.
weights.to_csv(
    OUTPUT / "njny_commute_zcta_weights_2021.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Number of connections between different regions:", len(weights))
print("Both weight checks passed. Output file saved.")
display(weights.head(10))

Number of connections between different regions: 788838
Both weight checks passed. Output file saved.


,origin_state,origin_zcta,destination_state,destination_zcta,commute_jobs,year,job_type,inbound_weight,outbound_weight
1,NJ,07001,NJ,07002,27,2021,JT00,0.002693,0.003899
2,NJ,07001,NJ,07003,17,2021,JT00,0.001498,0.002455
3,NJ,07001,NJ,07004,10,2021,JT00,0.000500,0.001444
4,NJ,07001,NJ,07005,2,2021,JT00,0.000481,0.000289
5,NJ,07001,NJ,07006,3,2021,JT00,0.000319,0.000433
6,NJ,07001,NJ,07008,123,2021,JT00,0.013135,0.017764
7,NJ,07001,NJ,07009,3,2021,JT00,0.000779,0.000433
8,NJ,07001,NJ,07011,4,2021,JT00,0.000532,0.000578
9,NJ,07001,NJ,07012,9,2021,JT00,0.001322,0.001300
10,NJ,07001,NJ,07013,5,2021,JT00,0.000565,0.000722
